In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint

In [2]:
# Datasets
datasets = {
    "heart": ("Data/heart_train.csv", "Data/heart_test.csv"),
    "diabetes": ("Data/diabetes_train.csv", "Data/diabetes_test.csv"),
    "cancer": ("Data/cancer_train.csv", "Data/cancer_test.csv"),
    "alzheimer": ("Data/alzheimer_train.csv", "Data/alzheimer_test.csv")
}

In [3]:
# Grid of hyperparameters 
param_uniform = {
    'max_depth': randint(3, 15),                    # tree depth
    'learning_rate': uniform(0.01, 0.29),           # eta: 0.01-0.3
    'n_estimators': randint(50, 500),               # number of trees
    'subsample': uniform(0.5, 0.5),                 # 0.5-1.0
    'colsample_bytree': uniform(0.5, 0.5),          # 0.5-1.0
    'gamma': uniform(0, 5),                         # min split loss
    'reg_alpha': uniform(0, 1),                     # L1 regularization
    'reg_lambda': uniform(0, 2),                    # L2 regularization
    'min_child_weight': randint(1, 10)              # minimum sum of instance weight
}

In [4]:
all_results = []

for name, (train_path, test_path) in datasets.items():
    print(f"Training: {name}")
    
    # Data load
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]
    
    # XGBoost model
    xgb = XGBClassifier(
        random_state=42,
        eval_metric='auc'  
    )

    
    # Random Search
    random_search = RandomizedSearchCV(
        estimator=xgb,
        param_distributions=param_uniform,
        n_iter=100,                    
        scoring='roc_auc',
        cv=5,                          
        random_state=42,
        n_jobs=-1
    )
    
    # Fit
    random_search.fit(X_train, y_train)
    
    # Result
    cv_results = pd.DataFrame(random_search.cv_results_)
    
    # Testing on test sets
    for i, params in enumerate(random_search.cv_results_['params']):
        # Training again on parameters from random_search.cv_results_ :(
        model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **params
        )
        model.fit(X_train, y_train)
        
        y_proba = model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
        
        all_results.append({
            "dataset": name,
            "params": params,
            "cv_roc_auc": cv_results.loc[i, 'mean_test_score'],
            "test_roc_auc": test_auc
        })
    
#Results
results_df = pd.DataFrame(all_results)

Training: heart
Training: diabetes
Training: cancer
Training: alzheimer


In [5]:
#Summary
for dataset in datasets.keys():
    dataset_results = results_df[results_df['dataset'] == dataset]
    best_idx = dataset_results['test_roc_auc'].idxmax()
    best_result = dataset_results.loc[best_idx]
    
    print(f"\n{dataset.upper()}:")
    print(f"  Best test AUC: {best_result['test_roc_auc']:.4f}")
    print(f"  CV AUC: {best_result['cv_roc_auc']:.4f}")
    print(f"  Parameters: {best_result['params']}")


HEART:
  Best test AUC: 0.8005
  CV AUC: 0.8026
  Parameters: {'colsample_bytree': 0.6774525952313604, 'gamma': 4.784004425632282, 'learning_rate': 0.20626327228304794, 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 416, 'reg_alpha': 0.08328441119525964, 'reg_lambda': 0.18340829451696217, 'subsample': 0.8012204629505595}

DIABETES:
  Best test AUC: 0.8411
  CV AUC: 0.8199
  Parameters: {'colsample_bytree': 0.8343216099622155, 'gamma': 4.646879945637929, 'learning_rate': 0.17146123897403964, 'max_depth': 10, 'min_child_weight': 3, 'n_estimators': 222, 'reg_alpha': 0.7694929331919369, 'reg_lambda': 0.37408749711504674, 'subsample': 0.6618396182021218}

CANCER:
  Best test AUC: 0.8721
  CV AUC: 0.8662
  Parameters: {'colsample_bytree': 0.5157145928433671, 'gamma': 3.182052056318902, 'learning_rate': 0.10116323451213473, 'max_depth': 6, 'min_child_weight': 5, 'n_estimators': 456, 'reg_alpha': 0.6044173792778172, 'reg_lambda': 1.0796821826033463, 'subsample': 0.6015306123673847}

A

In [6]:
#Finding the best set of hyperparameters for each dataset 
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False]).groupby("dataset", as_index=False).first()
)
params_df = best_per_dataset["params"].apply(pd.Series)

# Creating new set of hyperparameters from all datasets
mean_params = params_df.mean()

mean_params_dict = mean_params.to_dict()
for param in ["max_depth", "min_child_weight", "n_estimators"]:
    mean_params_dict[param] = int(round(mean_params_dict[param]))
mean_results = []

#Training with new hyperparameters on all datasets
for name, (train_path, test_path) in datasets.items():
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = XGBClassifier(
            random_state=42,
            eval_metric='auc',
            **mean_params_dict
        )
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "star_test_roc_auc": mean_auc
    })

mean_df = pd.DataFrame(mean_results)

In [7]:
# Creating new dataframe with all results

# Star means that this set of hyperparameters is a mean from the best 4 sets of hyperparameters, one for each set
params_df = results_df['params'].apply(pd.Series)
results_df = pd.concat([results_df.drop('params', axis=1), params_df], axis=1)

results_col = results_df[['cv_roc_auc','test_roc_auc']]
results_df = pd.concat([results_df.drop(['cv_roc_auc','test_roc_auc'],axis=1),results_col], axis=1)

results_df = results_df.merge(mean_df, on='dataset')
results_df['diff_from_star'] = results_df['star_test_roc_auc'] - results_df['test_roc_auc']
results_df

,dataset,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample,cv_roc_auc,test_roc_auc,star_test_roc_auc,diff_from_star
0,heart,0.687270,4.753572,0.222278,7.0,7.0,171.0,0.155995,0.116167,0.933088,0.807082,0.784046,0.774757,-0.009290
1,heart,0.800558,3.540363,0.015970,4.0,8.0,463.0,0.212339,0.363650,0.591702,0.810297,0.787052,0.774757,-0.012296
2,heart,0.652121,2.623782,0.135264,3.0,3.0,413.0,0.514234,1.184829,0.523225,0.807207,0.775106,0.774757,-0.000349
3,heart,0.803772,0.852621,0.028865,6.0,9.0,365.0,0.563288,0.770833,0.507983,0.809995,0.777394,0.774757,-0.002638
4,heart,0.615447,1.205127,0.208146,14.0,8.0,480.0,0.173365,0.782121,0.591118,0.798637,0.737442,0.774757,0.037314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,alzheimer,0.689411,2.285001,0.185148,10.0,5.0,206.0,0.229251,1.444505,0.860018,0.871094,0.852405,0.856776,0.004371
396,alzheimer,0.820574,3.469742,0.167390,13.0,5.0,197.0,0.181598,1.816901,0.791696,0.869915,0.856050,0.856776,0.000726
397,alzheimer,0.700426,2.310029,0.284712,14.0,8.0,495.0,0.100795,0.512031,0.863048,0.866920,0.853842,0.856776,0.002934
398,alzheimer,0.796481,0.511063,0.276438,7.0,8.0,342.0,0.707239,0.305078,0.788144,0.852435,0.836691,0.856776,0.020085


In [8]:
results_df.to_csv("Results/xgboost_uniform.csv", index=False)

In [11]:
# Best parameters for each dataset
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .drop(['star_test_roc_auc', 'diff_from_star'], axis=1)
)

# STAR row
mean_row = {
    "dataset": "STAR",
    **mean_params_dict,
    "cv_roc_auc": 0,
    "test_roc_auc": mean_df["star_test_roc_auc"].mean()
}

summary_df = pd.concat([best_per_dataset, pd.DataFrame([mean_row])], ignore_index=True)

In [12]:
summary_df.to_csv("Results/xgboost_uniform_summary.csv", index=False)

,dataset,colsample_bytree,gamma,learning_rate,max_depth,min_child_weight,n_estimators,reg_alpha,reg_lambda,subsample,cv_roc_auc,test_roc_auc
0,alzheimer,0.877681,2.125779,0.070303,6.0,6.0,240.0,0.842285,0.899508,0.697575,0.874153,0.866092
1,cancer,0.515715,3.182052,0.101163,6.0,5.0,456.0,0.604417,1.079682,0.601531,0.866213,0.872081
2,diabetes,0.834322,4.646880,0.171461,10.0,3.0,222.0,0.769493,0.374087,0.661840,0.819894,0.841075
3,heart,0.677453,4.784004,0.206263,6.0,3.0,416.0,0.083284,0.183408,0.801220,0.802644,0.800473
4,STAR,0.726292,3.684679,0.137298,7.0,4.0,334.0,0.574870,0.634172,0.690541,0.000000,0.833987
